# AI_EIARI4A_2026: Lab 0 - Baseline Transformer Internals
## 1 Million Parameter Transformer Baseline (TensorFlow/Keras)

**Objective:** In this lab, you will explore a small Transformer architecture, understand its parameter count, 
and perform fine-tuning on a custom text dataset using the TensorFlow/Keras framework.

### 1. Setup and Imports
We use TensorFlow's Keras API, consistent with the models developed earlier in the course.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers
import numpy as np
import os

print("TensorFlow version:", tf.__version__)

### 2. Building the 1M Parameter Architecture
This model uses a Decoder-style architecture. 
**Architecture details:**
- **Embedding Dimension:** 128
- **Transformer Blocks:** 2
- **Attention Heads:** 4
- **Feed Forward Dimension:** 512

In [ ]:
def build_mini_transformer(vocab_size, seq_len=128, embed_dim=128, num_heads=4, ff_dim=512):
    inputs = layers.Input(shape=(seq_len,))
    
    # 1. Token Embedding
    x = layers.Embedding(input_dim=vocab_size, output_dim=embed_dim)(inputs)
    
    # 2. Transformer Blocks
    for i in range(2):
        # Multi-Head Attention layer
        attention_output = layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=embed_dim, name=f"mha_{i}"
        )(x, x)
        x = layers.LayerNormalization(epsilon=1e-6)(x + attention_output)
        
        # Feed Forward Network
        ffn_output = layers.Dense(ff_dim, activation="relu")(x)
        ffn_output = layers.Dense(embed_dim)(ffn_output)
        x = layers.LayerNormalization(epsilon=1e-6)(x + ffn_output)
    
    # 3. Output layer (Predicting next token probability)
    outputs = layers.Dense(vocab_size, activation="softmax")(x)
    
    model = tf.keras.Model(inputs=inputs, outputs=outputs)
    return model

# Create the model with a restricted vocabulary for educational purposes
vocab_size = 2000
model = build_mini_transformer(vocab_size=vocab_size)

model.compile(optimizer="adam", loss="sparse_categorical_crossentropy")
model.summary()

### 3. The Data Pipeline
We use the `TextVectorization` layer to handle tokenization, consistent with Week 9.

In [ ]:
vectorize_layer = tf.keras.layers.TextVectorization(
    max_tokens=vocab_size,
    output_mode='int',
    output_sequence_length=129 # 128 inputs + 1 target
)

def prepare_dataset(texts, batch_size=32):
    ds = tf.data.Dataset.from_tensor_slices(texts)
    vectorize_layer.adapt(ds.batch(64))
    
    def split_input_target(chunk):
        return chunk[:, :-1], chunk[:, 1:]

    dataset = ds.batch(batch_size).map(vectorize_layer).map(split_input_target)
    return dataset.prefetch(tf.data.AUTOTUNE)

# For testing, we use dummy data. In the real lab, students will use their cleaned CSV/Text data.
sample_data = ["The Vaal University of Technology is a leading provider of high quality education."] * 50
dataset = prepare_dataset(sample_data)

### 4. Fine-Tuning Task
Students will take the 'Base' weights and fine-tune them. Here we demonstrate how to freeze layers.

In [ ]:
# Freeze the Attention Layers to simulate Transfer Learning
for layer in model.layers:
    if "mha" in layer.name:
        layer.trainable = False

model.compile(optimizer=tf.keras.optimizers.Adam(1e-4), loss="sparse_categorical_crossentropy")
print("Fine-tuning started...")
model.fit(dataset, epochs=5)

### 5. Generative Inference
This function implements the autoregressive generation loop.

In [ ]:
def generate(model, prompt, length=10):
    tokens = vectorize_layer([prompt])
    for _ in range(length):
        preds = model.predict(tokens, verbose=0)
        next_id = np.argmax(preds[0, -1, :])
        
        # Shift tokens for next prediction
        tokens_np = tokens.numpy()
        tokens_np = np.roll(tokens_np, -1)
        tokens_np[0, -1] = next_id
        tokens = tf.convert_to_tensor(tokens_np)
        
        word = vectorize_layer.get_vocabulary()[next_id]
        prompt += " " + word
    return prompt

print(generate(model, "the university"))